In [ ]:
monthly_trend_df= spark.sql("""WITH monthly AS (
    SELECT
        date_trunc('month', hour_ts) AS month_ts,
        SUM(current_load) AS month_total
    FROM electric_analysis_dev.transformer_hourly_analysis
    GROUP BY date_trunc('month', hour_ts)
),

trend AS (
    SELECT
        month_ts,
        month_total,
        LAG(month_total) OVER (ORDER BY month_ts) AS prev_month_total
    FROM monthly
)

SELECT
    month_ts,
    ROUND(month_total, 2) AS month_total,
    ROUND(prev_month_total, 2) AS prev_month_total,
    ROUND(
        ((month_total - prev_month_total) / prev_month_total) * 100,
        2
    ) AS pct_change,

    CASE
        WHEN month_total > prev_month_total THEN 'Increase'
        WHEN month_total < prev_month_total THEN 'Decrease'
        ELSE 'No Change'
    END AS trend

FROM trend
ORDER BY month_ts;
""")

spark.sql("create database if not exists `electric_analysis_dev`")


monthly_trend_df.coalesce(1).write \
    .mode("overwrite") \
    .format("parquet") \
    .option("path", "s3://ops-autopilot-data/transformed/monthly_trend/") \
    .saveAsTable("`electric_analysis_dev`.`monthly_trend`")